In [265]:
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import kstest, shapiro, normaltest

In [266]:
import pymysql
import pandas as pd
with pymysql.connect(
    host="vmlp.pwdlearning.cloud",
    user="student_ds",
    password="@studentDS#2026",
    database="dekoruma",
    port=3306
) as connection:
    df = pd.read_sql("SELECT * FROM customers;", connection)
df

/var/folders/bb/_2z7z6bs4qbghjqy4r44sxkc0000gn/T/ipykernel_92981/1846524894.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM customers;", connection)


,customer_id,domicile_city,registration_date,registration_channel,age_group,membership_tier,phone_verified,email_verified,acquisition_channel
0,CUST00015LM0,None,2023-08-27 09:00:00,Physical Store,25-34,Regular,1,0,Organic
1,CUST0004NNEE,Pontianak,2023-08-24 14:00:00,Web,25-34,Regular,1,1,Paid Ads
2,CUST000570C3,Semarang,2023-08-12 14:00:00,Mobile App,35-44,Regular,1,1,Paid Ads
3,CUST0005VSDE,Yogyakarta,2023-06-18 17:00:00,Physical Store,25-34,Regular,1,1,Organic
4,CUST00097ZA9,Pontianak,2023-11-08 23:00:00,Physical Store,45-54,Regular,1,1,Walk-in
...,...,...,...,...,...,...,...,...,...
277923,CUSTZZZAH07L,Surabaya,2023-12-14 20:00:00,Web,25-34,Regular,1,1,Organic
277924,CUSTZZZB3IRL,Semarang,2023-06-16 18:00:00,Mobile App,25-34,Regular,1,1,Paid Ads
277925,CUSTZZZTMWY5,Semarang,2023-02-13 13:00:00,None,25-34,Regular,1,1,None
277926,CUSTZZZXX3RK,Denpasar,2023-09-12 00:00:00,Mobile App,25-34,Regular,1,1,Paid Ads


In [267]:
df_cust = df.copy()

In [268]:
df_cust.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277928 entries, 0 to 277927
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   customer_id           277928 non-null  object        
 1   domicile_city         250035 non-null  object        
 2   registration_date     277928 non-null  datetime64[ns]
 3   registration_channel  196501 non-null  object        
 4   age_group             277928 non-null  object        
 5   membership_tier       277928 non-null  object        
 6   phone_verified        277928 non-null  int64         
 7   email_verified        277928 non-null  int64         
 8   acquisition_channel   196501 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(6)
memory usage: 19.1+ MB


In [269]:
df_cust.describe()

,registration_date,phone_verified,email_verified
count,277928,277928.000000,277928.000000
mean,2023-07-07 09:02:36.757146112,0.878332,0.846309
min,2023-01-01 00:00:00,0.000000,0.000000
25%,2023-04-07 19:00:00,1.000000,1.000000
50%,2023-07-09 19:00:00,1.000000,1.000000
75%,2023-10-06 17:00:00,1.000000,1.000000
max,2023-12-31 23:00:00,1.000000,1.000000
std,NaN,0.326903,0.360653


In [270]:
df_cust.describe(include='object')

,customer_id,domicile_city,registration_channel,age_group,membership_tier,acquisition_channel
count,277928,250035,196501,277928,277928,196501
unique,277928,20,3,5,2,4
top,CUST00015LM0,Kemang,Physical Store,25-34,Regular,Organic
freq,1,17372,97802,94474,239299,58385


In [271]:
df_cust.isna().sum()

customer_id                 0
domicile_city           27893
registration_date           0
registration_channel    81427
age_group                   0
membership_tier             0
phone_verified              0
email_verified              0
acquisition_channel     81427
dtype: int64

In [272]:
df_cust[df_cust["acquisition_channel"].isnull()]

,customer_id,domicile_city,registration_date,registration_channel,age_group,membership_tier,phone_verified,email_verified,acquisition_channel
8,CUST000K4472,Alam Sutera,2023-04-02 23:00:00,None,55+,Dekoruma Plus,1,1,None
14,CUST001K3M4D,Surabaya,2023-01-21 17:00:00,None,45-54,Regular,1,1,None
16,CUST001XQ850,Denpasar,2023-04-24 22:00:00,None,45-54,Regular,1,1,None
22,CUST002BL5P0,Yogyakarta,2023-04-28 00:00:00,None,35-44,Regular,1,1,None
25,CUST002FEVLA,Padang,2023-04-15 16:00:00,None,55+,Regular,1,1,None
...,...,...,...,...,...,...,...,...,...
277904,CUSTZZVYXR08,Malang,2023-05-07 23:00:00,None,45-54,Regular,1,1,None
277909,CUSTZZWV91RS,Pontianak,2023-05-01 06:00:00,None,18-24,Regular,1,0,None
277916,CUSTZZY8X16R,Denpasar,2023-01-06 19:00:00,None,45-54,Regular,1,1,None
277918,CUSTZZYID53X,Depok,2023-01-06 21:00:00,None,45-54,Regular,1,1,None


In [273]:
((df_cust.isnull().sum() / len(df_cust)) * 100).round(2)

customer_id              0.00
domicile_city           10.04
registration_date        0.00
registration_channel    29.30
age_group                0.00
membership_tier          0.00
phone_verified           0.00
email_verified           0.00
acquisition_channel     29.30
dtype: float64

In [274]:
df_cust['domicile_city'].value_counts()

domicile_city
Kemang           17372
Kelapa Gading    17250
Bandung          17212
Medan            17211
Bekasi           17204
Alam Sutera      17096
Surabaya         17046
Bintaro          16972
Pontianak         9523
Balikpapan        9473
Semarang          9472
Solo              9455
Palembang         9454
Denpasar          9425
Padang            9393
Malang            9390
Tangerang         9330
Depok             9314
Yogyakarta        9239
Makassar          9204
Name: count, dtype: int64

In [275]:
df_cust['domicile_city'] = df_cust['domicile_city'].fillna('Kemang')

In [276]:
df_cust.isna().sum()

customer_id                 0
domicile_city               0
registration_date           0
registration_channel    81427
age_group                   0
membership_tier             0
phone_verified              0
email_verified              0
acquisition_channel     81427
dtype: int64

In [277]:
df_cust['registration_channel'].value_counts()

registration_channel
Physical Store    97802
Mobile App        55159
Web               43540
Name: count, dtype: int64

In [278]:
df_cust['registration_channel'] = df_cust['registration_channel'].fillna('Physical Store')

In [279]:
df_cust['acquisition_channel'].value_counts()

acquisition_channel
Organic     58385
Walk-in     54076
Referral    44212
Paid Ads    39828
Name: count, dtype: int64

In [280]:
df_cust['acquisition_channel'] = df_cust['acquisition_channel'].fillna('Organic')

In [281]:
df_cust.isna().sum()

customer_id             0
domicile_city           0
registration_date       0
registration_channel    0
age_group               0
membership_tier         0
phone_verified          0
email_verified          0
acquisition_channel     0
dtype: int64

In [282]:
df_cust.to_csv('/Users/gustafdavid/Desktop/Purwadika/capston 2/customertab.csv', index=False)

In [283]:
import pymysql
import pandas as pd
with pymysql.connect(
    host="vmlp.pwdlearning.cloud",
    user="student_ds",
    password="@studentDS#2026",
    database="dekoruma",
    port=3306
) as connection:
    dfo = pd.read_sql("SELECT * FROM orders;", connection)

df_ordr = dfo.copy()
print (df_ordr)


/var/folders/bb/_2z7z6bs4qbghjqy4r44sxkc0000gn/T/ipykernel_92981/3336111784.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dfo = pd.read_sql("SELECT * FROM orders;", connection)


                  order_id store_id          sku_id   customer_id  \
0       DKR/230101/1013407  DKR-006  SKU-FURN-00144  CUSTIBZ66NOP   
1       DKR/230101/1039548  DKR-009  SKU-FURN-00737  CUST8VYFWXJW   
2       DKR/230101/1114362  DKR-039  SKU-FURN-01395  CUSTGAGULSK4   
3       DKR/230101/1116644  DKR-027  SKU-FURN-01194  CUSTLSVROHVO   
4       DKR/230101/1132206  DKR-003  SKU-FURN-00679  CUST7F0EZ9R2   
...                    ...      ...             ...           ...   
321755  DKR/231231/9941531  DKR-025  SKU-FURN-01497  CUST0NGXRF0E   
321756  DKR/231231/9952013  DKR-016  SKU-FURN-01362  CUST6PA5XBOD   
321757  DKR/231231/9969766  DKR-018  SKU-FURN-00547  CUSTBEXLJEEW   
321758  DKR/231231/9974865  DKR-029  SKU-FURN-00506  CUSTZ3KWPZ22   
321759  DKR/231231/9999385  DKR-005  SKU-FURN-00901  CUSTXBV3NL4A   

                order_date       delivery_date    installer_id  \
0      2023-01-01 20:00:00 2023-01-03 20:00:00  TUKANG-JKT-654   
1      2023-01-01 14:00:00 2023-01-03 1

In [284]:
print (df_ordr.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321760 entries, 0 to 321759
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             321760 non-null  object        
 1   store_id             321760 non-null  object        
 2   sku_id               321760 non-null  object        
 3   customer_id          321760 non-null  object        
 4   order_date           321760 non-null  datetime64[ns]
 5   delivery_date        321760 non-null  datetime64[ns]
 6   installer_id         204557 non-null  object        
 7   installation_date    204557 non-null  datetime64[ns]
 8   installation_status  321760 non-null  object        
 9   assembly_fee         321760 non-null  int64         
dtypes: datetime64[ns](3), int64(1), object(6)
memory usage: 24.5+ MB
None


In [285]:
df_ordr['Install_time'] = (
    df_ordr['installation_date'] - df_ordr['delivery_date'])

In [286]:
df_ordr['delivery_time'] = (
    df_ordr['delivery_date'] - df_ordr['order_date'])

In [287]:
df_ordr.describe()

,order_date,delivery_date,installation_date,assembly_fee,Install_time,delivery_time
count,321760,321760,204557,3.217600e+05,204557,321760
mean,2023-07-11 16:00:18.337892096,2023-07-16 04:03:06.701889280,2023-07-18 15:05:39.784021248,1.955686e+05,2 days 10:17:31.681438425,4 days 12:02:48.363998010
min,2023-01-01 09:00:00,2023-01-03 09:00:00,2023-01-01 09:00:00,-1.500000e+05,-3 days +00:00:00,2 days 00:00:00
25%,2023-04-13 16:00:00,2023-04-18 10:00:00,2023-04-20 17:00:00,0.000000e+00,1 days 00:00:00,3 days 00:00:00
50%,2023-07-09 17:00:00,2023-07-14 11:00:00,2023-07-16 13:00:00,6.567700e+04,2 days 00:00:00,5 days 00:00:00
75%,2023-10-11 18:00:00,2023-10-16 12:00:00,2023-10-18 17:00:00,1.515260e+05,4 days 00:00:00,6 days 00:00:00
max,2023-12-31 20:00:00,2024-01-07 20:00:00,2024-01-13 18:00:00,9.990000e+07,6 days 00:00:00,7 days 00:00:00
std,NaN,NaN,NaN,3.310378e+06,1 days 19:14:06.421976149,1 days 12:33:04.330189949


In [288]:
df_ordr['delivery_time'].value_counts()

delivery_time
4 days    71698
5 days    71297
6 days    51059
3 days    50624
7 days    38576
2 days    38506
Name: count, dtype: int64

In [289]:
df_ordr['Install_time'].value_counts()

Install_time
3 days     42133
2 days     41913
4 days     31178
1 days     30964
0 days     26249
5 days     17050
6 days      9484
-2 days     1879
-3 days     1877
-1 days     1830
Name: count, dtype: int64

ada pengisian data yang tidak benar -> ada pengisian tanggal instalasi mendahului tanggal pengiriman

In [290]:
df_ordr.describe(include='object')

,order_id,store_id,sku_id,customer_id,installer_id,installation_status
count,321760,321760,321760,321760,204557,321760
unique,321760,40,1500,134274,1658,5
top,DKR/230101/1013407,DKR-039,SKU-FURN-00646,CUSTB2Y2B20Z,TUKANG-JKT-990,Success
freq,1,20986,266,29,166,175775


In [291]:
df_ordr.isnull().sum()

order_id                    0
store_id                    0
sku_id                      0
customer_id                 0
order_date                  0
delivery_date               0
installer_id           117203
installation_date      117203
installation_status         0
assembly_fee                0
Install_time           117203
delivery_time               0
dtype: int64

In [292]:
((df_ordr.isnull().sum()/ len(df_ordr)*100)).round(2)

order_id                0.00
store_id                0.00
sku_id                  0.00
customer_id             0.00
order_date              0.00
delivery_date           0.00
installer_id           36.43
installation_date      36.43
installation_status     0.00
assembly_fee            0.00
Install_time           36.43
delivery_time           0.00
dtype: float64

In [338]:
df_ordr['installation_status'].value_counts()

installation_status
Success                   175775
No Assembly Required       81252
No Assembly Service        35951
Failed - Missing Parts     18570
Failed - Damaged Panel     10212
Name: count, dtype: int64

In [293]:
df_ordr.to_csv('/Users/gustafdavid/Desktop/Purwadika/capston 2/orders.csv', index=False)

In [294]:
import pymysql
import pandas as pd
with pymysql.connect(
    host="vmlp.pwdlearning.cloud",
    user="student_ds",
    password="@studentDS#2026",
    database="dekoruma",
    port=3306
) as connection:
    dfp = pd.read_sql("SELECT * FROM products;", connection)

df_prod = dfp.copy()
print (df_prod)

              sku_id                product_name      category  \
0     SKU-FURN-00001    Furnitur Kayu Olahan 407  Office Chair   
1     SKU-FURN-00002    Furnitur Kayu Olahan 536      Wardrobe   
2     SKU-FURN-00003  Sofa Japandi Minimalis 867          Sofa   
3     SKU-FURN-00004    Furnitur Kayu Olahan 727   Kursi Kerja   
4     SKU-FURN-00005    Furnitur Kayu Olahan 363      Wardrobe   
...              ...                         ...           ...   
1495  SKU-FURN-01496  Sofa Japandi Minimalis 932          Sofa   
1496  SKU-FURN-01497    Furnitur Kayu Olahan 956          lmri   
1497  SKU-FURN-01498    Furnitur Kayu Olahan 774      Wardrobe   
1498  SKU-FURN-01499    Furnitur Kayu Olahan 642  Dining Table   
1499  SKU-FURN-01500    Furnitur Kayu Olahan 559  Office Chair   

      is_assembly_required     supplier_id  unit_price product_family  \
0                        1            None     1472960          Kursi   
1                        1  SUP-PBR-SDA-05     2212127       

/var/folders/bb/_2z7z6bs4qbghjqy4r44sxkc0000gn/T/ipykernel_92981/3590934324.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dfp = pd.read_sql("SELECT * FROM products;", connection)


In [295]:
df_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sku_id                1500 non-null   object
 1   product_name          1500 non-null   object
 2   category              1500 non-null   object
 3   is_assembly_required  1500 non-null   int64 
 4   supplier_id           1442 non-null   object
 5   unit_price            1500 non-null   int64 
 6   product_family        1500 non-null   object
 7   weight_category       1500 non-null   object
dtypes: int64(2), object(6)
memory usage: 93.9+ KB


In [296]:
df_prod['weight_category'].value_counts()

weight_category
Bulky        525
Sedang       450
Oversized    294
Ringan       231
Name: count, dtype: int64

In [297]:
df_prod.describe()

,is_assembly_required,unit_price
count,1500.000000,1.500000e+03
mean,0.746000,2.182266e+06
std,0.435443,1.263119e+06
min,0.000000,1.605010e+05
25%,0.000000,1.148108e+06
50%,1.000000,2.067152e+06
75%,1.000000,3.218430e+06
max,1.000000,5.494205e+06


In [298]:
df_prod.describe(include=["object",'category'])

,sku_id,product_name,category,supplier_id,product_family,weight_category
count,1500,1500,1500,1442,1500,1500
unique,1500,918,11,32,6,4
top,SKU-FURN-00001,Furnitur Kayu Olahan 250,Dekorasi,SUP-PBR-BKS-05,Lemari,Bulky
freq,1,7,215,119,348,525


In [299]:
df_prod.isnull().sum()

sku_id                   0
product_name             0
category                 0
is_assembly_required     0
supplier_id             58
unit_price               0
product_family           0
weight_category          0
dtype: int64

In [300]:
df_prod[df_prod['supplier_id'].isnull()]

,sku_id,product_name,category,is_assembly_required,supplier_id,unit_price,product_family,weight_category
0,SKU-FURN-00001,Furnitur Kayu Olahan 407,Office Chair,1,None,1472960,Kursi,Sedang
14,SKU-FURN-00015,Furnitur Kayu Olahan 880,Kursi Kerja,1,None,1338206,Kursi,Ringan
19,SKU-FURN-00020,Furnitur Kayu Olahan 453,Office Chair,1,None,709200,Kursi,Sedang
22,SKU-FURN-00023,Furnitur Kayu Olahan 233,Wardrobe,1,None,2888049,Lemari,Bulky
25,SKU-FURN-00026,Furnitur Kayu Olahan 238,Kursi Kerja,1,None,960270,Kursi,Ringan
36,SKU-FURN-00037,Hiasan Meja / Karpet Aesthetic 65,Dekorasi,0,None,439807,Dekorasi,Ringan
50,SKU-FURN-00051,Sofa Japandi Minimalis 525,Sofa,1,None,4416896,Sofa,Bulky
57,SKU-FURN-00058,Furnitur Kayu Olahan 650,Kursi Kerja,1,None,1173571,Kursi,Sedang
58,SKU-FURN-00059,Furnitur Kayu Olahan 138,Dining Table,1,None,1457066,Meja,Sedang
59,SKU-FURN-00060,Furnitur Kayu Olahan 303,Wardrobe,1,None,2920427,Lemari,Oversized


In [301]:
((df_prod.isnull().sum() / len(df_prod)) * 100).round(2)

sku_id                  0.00
product_name            0.00
category                0.00
is_assembly_required    0.00
supplier_id             3.87
unit_price              0.00
product_family          0.00
weight_category         0.00
dtype: float64

In [302]:
#df_prod = df_prod.dropna(subset=['supplier_id'])

In [303]:
df_prod.to_csv('/Users/gustafdavid/Desktop/Purwadika/capston 2/products.csv', index=False)

In [304]:
with pymysql.connect(
    host="vmlp.pwdlearning.cloud",
    user="student_ds",
    password="@studentDS#2026",
    database="dekoruma",
    port=3306
) as connection:
    dfr = pd.read_sql("SELECT * FROM return_requests;", connection)

df_retur = dfr.copy()
print (df_retur)

/var/folders/bb/_2z7z6bs4qbghjqy4r44sxkc0000gn/T/ipykernel_92981/1426131149.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dfr = pd.read_sql("SELECT * FROM return_requests;", connection)


                 return_id            order_id  \
0      RET-DKR/230106/2001  DKR/230101/3483685   
1      RET-DKR/230106/2333  DKR/230101/9221271   
2      RET-DKR/230106/6830  DKR/230101/8473576   
3      RET-DKR/230107/1613  DKR/230103/6898888   
4      RET-DKR/230107/3963  DKR/230101/4207734   
...                    ...                 ...   
24838  RET-DKR/240120/1625  DKR/231230/8096784   
24839  RET-DKR/240120/7129  DKR/231229/6654327   
24840  RET-DKR/240121/2278  DKR/231221/3302875   
24841  RET-DKR/240121/2993  DKR/231231/9425435   
24842  RET-DKR/240123/8538  DKR/231231/4764857   

                             return_reason return_request_date  \
0      Failed Installation - Missing Parts 2023-01-06 09:00:00   
1      Failed Installation - Missing Parts 2023-01-06 15:00:00   
2      Failed Installation - Damaged Panel 2023-01-06 11:00:00   
3      Failed Installation - Damaged Panel 2023-01-07 13:00:00   
4      Failed Installation - Damaged Panel 2023-01-07 17:00:00   
...

In [305]:
df_retur.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24843 entries, 0 to 24842
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   return_id              24843 non-null  object        
 1   order_id               24843 non-null  object        
 2   return_reason          24843 non-null  object        
 3   return_request_date    24843 non-null  datetime64[ns]
 4   resolution_status      24843 non-null  object        
 5   processing_days        20991 non-null  float64       
 6   refund_method          18919 non-null  object        
 7   refund_amount          20991 non-null  float64       
 8   restocking_fee_amount  24843 non-null  int64         
 9   satisfaction_score     18919 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(5)
memory usage: 1.9+ MB


In [306]:
df_retur['processing_days'] = df_retur['processing_days'].astype('Int64')

In [307]:
df_retur.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24843 entries, 0 to 24842
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   return_id              24843 non-null  object        
 1   order_id               24843 non-null  object        
 2   return_reason          24843 non-null  object        
 3   return_request_date    24843 non-null  datetime64[ns]
 4   resolution_status      24843 non-null  object        
 5   processing_days        20991 non-null  Int64         
 6   refund_method          18919 non-null  object        
 7   refund_amount          20991 non-null  float64       
 8   restocking_fee_amount  24843 non-null  int64         
 9   satisfaction_score     18919 non-null  float64       
dtypes: Int64(1), datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 1.9+ MB


In [308]:
df_retur.describe()

,return_request_date,processing_days,refund_amount,restocking_fee_amount,satisfaction_score
count,24843,20991.0,2.099100e+04,24843.000000,18919.000000
mean,2023-07-21 22:36:18.867286272,4.004383,2.224646e+06,5409.342672,3.779851
min,2023-01-06 09:00:00,1.0,-1.500000e+05,0.000000,1.000000
25%,2023-04-23 19:00:00,2.0,1.191900e+06,0.000000,3.000000
50%,2023-07-19 11:00:00,3.0,1.888000e+06,0.000000,4.000000
75%,2023-10-22 12:30:00,5.0,2.906300e+06,0.000000,4.000000
max,2024-01-23 14:00:00,19.0,1.053163e+08,271800.000000,5.000000
std,NaN,2.498571,4.807218e+06,27644.723604,0.746705


In [309]:
df_retur.describe(include = ['object', 'category'])

,return_id,order_id,return_reason,resolution_status,refund_method
count,24843,24843,24843,24843,18919
unique,24843,24843,5,3,3
top,RET-DKR/230106/2001,DKR/230101/3483685,Failed Installation - Missing Parts,Selesai,Refund ke Rekening
freq,1,1,12637,18919,8863


In [310]:
df_retur.isnull().sum()

return_id                   0
order_id                    0
return_reason               0
return_request_date         0
resolution_status           0
processing_days          3852
refund_method            5924
refund_amount            3852
restocking_fee_amount       0
satisfaction_score       5924
dtype: int64

In [311]:
(df_retur.isnull().sum()/len(df_retur)*100).round(2)

return_id                 0.00
order_id                  0.00
return_reason             0.00
return_request_date       0.00
resolution_status         0.00
processing_days          15.51
refund_method            23.85
refund_amount            15.51
restocking_fee_amount     0.00
satisfaction_score       23.85
dtype: float64

In [312]:
df_retur.to_csv('/Users/gustafdavid/Desktop/Purwadika/capston 2/retur.csv', index=False)

In [313]:
with pymysql.connect(
    host="vmlp.pwdlearning.cloud",
    user="student_ds",
    password="@studentDS#2026",
    database="dekoruma",
    port=3306
) as connection:
    dfs = pd.read_sql("SELECT * FROM stores;", connection)

df_store = dfs.copy()
print (df_store)

   store_id            store_name    channel_type           city store_tier  \
0   DKR-001     DEC Alam Sutera 1  Physical Store    Alam Sutera   Standard   
1   DKR-002         DEC Bintaro 2  Physical Store        Bintaro       Mini   
2   DKR-003          DEC Kemang 3  Physical Store         Kemang   Standard   
3   DKR-004   DEC Kelapa Gading 4  Physical Store  Kelapa Gading   Standard   
4   DKR-005           DEC Medan 5  Physical Store          Medan       Mini   
5   DKR-006          DEC Kemang 6  Physical Store         Kemang   Standard   
6   DKR-007        DEC Surabaya 7  Physical Store       Surabaya   Flagship   
7   DKR-008         DEC Bandung 8  Physical Store        Bandung       Mini   
8   DKR-009   DEC Kelapa Gading 9  Physical Store  Kelapa Gading   Standard   
9   DKR-010         DEC Kemang 10  Physical Store         Kemang   Standard   
10  DKR-011  DEC Kelapa Gading 11  Physical Store  Kelapa Gading   Flagship   
11  DKR-012        DEC Bintaro 12  Physical Store   

/var/folders/bb/_2z7z6bs4qbghjqy4r44sxkc0000gn/T/ipykernel_92981/300355064.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dfs = pd.read_sql("SELECT * FROM stores;", connection)


In [314]:
df_store.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   store_id      40 non-null     object
 1   store_name    40 non-null     object
 2   channel_type  40 non-null     object
 3   city          38 non-null     object
 4   store_tier    38 non-null     object
 5   opening_date  40 non-null     object
dtypes: object(6)
memory usage: 2.0+ KB


In [315]:
df_store.describe(include="object")

,store_id,store_name,channel_type,city,store_tier,opening_date
count,40,40,40,38,38,40
unique,40,40,3,8,3,39
top,DKR-001,DEC Alam Sutera 1,Physical Store,Alam Sutera,Standard,2020-09-18
freq,1,1,38,8,23,2


In [316]:
df_store.isnull().sum()

store_id        0
store_name      0
channel_type    0
city            2
store_tier      2
opening_date    0
dtype: int64

In [317]:
(df_store.isnull().sum()/len(df_store)*100).round(2)

store_id        0.0
store_name      0.0
channel_type    0.0
city            5.0
store_tier      5.0
opening_date    0.0
dtype: float64

In [318]:
df_store.to_csv('/Users/gustafdavid/Desktop/Purwadika/capston 2/stores.csv', index=False)

In [319]:
with pymysql.connect(
    host="vmlp.pwdlearning.cloud",
    user="student_ds",
    password="@studentDS#2026",
    database="dekoruma",
    port=3306
) as connection:
    dfsp = pd.read_sql("SELECT * FROM suppliers;", connection)

df_sup = dfsp.copy()
print (df_sup)

/var/folders/bb/_2z7z6bs4qbghjqy4r44sxkc0000gn/T/ipykernel_92981/193748257.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dfsp = pd.read_sql("SELECT * FROM suppliers;", connection)


       supplier_id             supplier_name  supplier_type          region  \
0   SUP-IMP-JKT-01       Nordic Home Trading       Importir   Jakarta/Priok   
1   SUP-IMP-SBY-01        Global Home Import       Importir  Surabaya/Perak   
2   SUP-PBR-BKS-01  Prima Wood Manufacturing         Pabrik          Bekasi   
3   SUP-PBR-BKS-02         Trimatra Furnindo         Pabrik          Bekasi   
4   SUP-PBR-BKS-03    Mega Furnitur Industri         Pabrik          Bekasi   
5   SUP-PBR-BKS-04        Andalan Kayu Prima         Pabrik          Bekasi   
6   SUP-PBR-BKS-05          Sumber Kayu Jaya         Pabrik          Bekasi   
7   SUP-PBR-SDA-01      Bina Furnitur Sejati         Pabrik        Sidoarjo   
8   SUP-PBR-SDA-02     Sinar Mebel Nusantara         Pabrik        Sidoarjo   
9   SUP-PBR-SDA-03   Karunia Wood Industries         Pabrik        Sidoarjo   
10  SUP-PBR-SDA-04     Karya Mebel Sejahtera         Pabrik        Sidoarjo   
11  SUP-PBR-SDA-05   Lestari Kayu Manufaktur        

In [320]:
df_sup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   supplier_id                   32 non-null     object
 1   supplier_name                 32 non-null     object
 2   supplier_type                 32 non-null     object
 3   region                        32 non-null     object
 4   primary_family                32 non-null     object
 5   secondary_family              18 non-null     object
 6   quality_certification_status  32 non-null     object
 7   partnership_start_date        32 non-null     object
dtypes: object(8)
memory usage: 2.1+ KB


In [321]:
df_sup.isnull().sum()

supplier_id                      0
supplier_name                    0
supplier_type                    0
region                           0
primary_family                   0
secondary_family                14
quality_certification_status     0
partnership_start_date           0
dtype: int64

In [322]:
(df_sup.isnull().sum()/len(df_sup)*100).round(2)

supplier_id                      0.00
supplier_name                    0.00
supplier_type                    0.00
region                           0.00
primary_family                   0.00
secondary_family                43.75
quality_certification_status     0.00
partnership_start_date           0.00
dtype: float64

In [323]:
df_sup.to_csv('/Users/gustafdavid/Desktop/Purwadika/capston 2/suppliers.csv', index=False)

In [324]:
df_gabung = (
    df_ordr
    .merge(df_store, on='store_id', how='left')
    .merge(df_prod, on='sku_id', how='left')
    .merge(df_cust, on='customer_id', how='left')
    .merge(df_sup, on='supplier_id', how='left')
    .merge(df_retur, on='order_id', how='left')
)

In [325]:
df_gabung

,order_id,store_id,sku_id,customer_id,order_date,delivery_date,installer_id,installation_date,installation_status,assembly_fee,...,partnership_start_date,return_id,return_reason,return_request_date,resolution_status,processing_days,refund_method,refund_amount,restocking_fee_amount,satisfaction_score
0,DKR/230101/1013407,DKR-006,SKU-FURN-00144,CUSTIBZ66NOP,2023-01-01 20:00:00,2023-01-03 20:00:00,TUKANG-JKT-654,2023-01-09 20:00:00,Success,67719,...,2017-05-07,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN
1,DKR/230101/1039548,DKR-009,SKU-FURN-00737,CUST8VYFWXJW,2023-01-01 14:00:00,2023-01-03 14:00:00,None,NaT,No Assembly Service,0,...,2024-01-05,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN
2,DKR/230101/1114362,DKR-039,SKU-FURN-01395,CUSTGAGULSK4,2023-01-01 19:00:00,2023-01-08 19:00:00,TUKANG-BDG-039,2023-01-12 19:00:00,Success,149903,...,2023-02-03,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN
3,DKR/230101/1116644,DKR-027,SKU-FURN-01194,CUSTLSVROHVO,2023-01-01 12:00:00,2023-01-06 12:00:00,None,NaT,No Assembly Service,0,...,2020-05-29,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN
4,DKR/230101/1132206,DKR-003,SKU-FURN-00679,CUST7F0EZ9R2,2023-01-01 10:00:00,2023-01-05 10:00:00,TUKANG-JKT-859,2023-01-07 10:00:00,Success,141102,...,2021-02-15,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321755,DKR/231231/9941531,DKR-025,SKU-FURN-01497,CUST0NGXRF0E,2023-12-31 19:00:00,2024-01-07 19:00:00,None,NaT,No Assembly Service,0,...,2023-12-21,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN
321756,DKR/231231/9952013,DKR-016,SKU-FURN-01362,CUST6PA5XBOD,2023-12-31 20:00:00,2024-01-04 20:00:00,TUKANG-JKT-435,2024-01-07 20:00:00,Success,181074,...,2020-02-18,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN
321757,DKR/231231/9969766,DKR-018,SKU-FURN-00547,CUSTBEXLJEEW,2023-12-31 09:00:00,2024-01-04 09:00:00,None,NaT,No Assembly Service,0,...,2019-03-05,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN
321758,DKR/231231/9974865,DKR-029,SKU-FURN-00506,CUSTZ3KWPZ22,2023-12-31 13:00:00,2024-01-07 13:00:00,TUKANG-JKT-794,2024-01-13 13:00:00,Success,66757,...,2021-08-03,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaN,NaN


In [327]:
df_gabung.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321760 entries, 0 to 321759
Data columns (total 48 columns):
 #   Column                        Non-Null Count   Dtype          
---  ------                        --------------   -----          
 0   order_id                      321760 non-null  object         
 1   store_id                      321760 non-null  object         
 2   sku_id                        321760 non-null  object         
 3   customer_id                   321760 non-null  object         
 4   order_date                    321760 non-null  datetime64[ns] 
 5   delivery_date                 321760 non-null  datetime64[ns] 
 6   installer_id                  204557 non-null  object         
 7   installation_date             204557 non-null  datetime64[ns] 
 8   installation_status           321760 non-null  object         
 9   assembly_fee                  321760 non-null  int64          
 10  Install_time                  204557 non-null  timedelta64[ns]
 11  

In [328]:
df_gabung.describe()

,order_date,delivery_date,installation_date,assembly_fee,Install_time,delivery_time,is_assembly_required,unit_price,registration_date,phone_verified,email_verified,return_request_date,processing_days,refund_amount,restocking_fee_amount,satisfaction_score
count,321760,321760,204557,3.217600e+05,204557,321760,321760.000000,3.217600e+05,321760,321760.000000,321760.00000,24843,20991.0,2.099100e+04,24843.000000,18919.000000
mean,2023-07-11 16:00:18.337892096,2023-07-16 04:03:06.701889280,2023-07-18 15:05:39.784021248,1.955686e+05,2 days 10:17:31.681438425,4 days 12:02:48.363998010,0.747476,2.182215e+06,2023-04-09 03:47:13.075584512,0.892323,0.86521,2023-07-21 22:36:18.867287040,4.004383,2.224646e+06,5409.342672,3.779851
min,2023-01-01 09:00:00,2023-01-03 09:00:00,2023-01-01 09:00:00,-1.500000e+05,-3 days +00:00:00,2 days 00:00:00,0.000000,1.605010e+05,2023-01-01 00:00:00,0.000000,0.00000,2023-01-06 09:00:00,1.0,-1.500000e+05,0.000000,1.000000
25%,2023-04-13 16:00:00,2023-04-18 10:00:00,2023-04-20 17:00:00,0.000000e+00,1 days 00:00:00,3 days 00:00:00,0.000000,1.149741e+06,2023-01-30 04:00:00,1.000000,1.00000,2023-04-23 19:00:00,2.0,1.191900e+06,0.000000,3.000000
50%,2023-07-09 17:00:00,2023-07-14 11:00:00,2023-07-16 13:00:00,6.567700e+04,2 days 00:00:00,5 days 00:00:00,1.000000,2.068405e+06,2023-03-17 15:00:00,1.000000,1.00000,2023-07-19 11:00:00,3.0,1.888000e+06,0.000000,4.000000
75%,2023-10-11 18:00:00,2023-10-16 12:00:00,2023-10-18 17:00:00,1.515260e+05,4 days 00:00:00,6 days 00:00:00,1.000000,3.217828e+06,2023-05-30 16:00:00,1.000000,1.00000,2023-10-22 12:30:00,5.0,2.906300e+06,0.000000,4.000000
max,2023-12-31 20:00:00,2024-01-07 20:00:00,2024-01-13 18:00:00,9.990000e+07,6 days 00:00:00,7 days 00:00:00,1.000000,5.494205e+06,2023-12-31 21:00:00,1.000000,1.00000,2024-01-23 14:00:00,19.0,1.053163e+08,271800.000000,5.000000
std,NaN,NaN,NaN,3.310378e+06,1 days 19:14:06.421976149,1 days 12:33:04.330189949,0.434461,1.262537e+06,NaN,0.309972,0.34150,NaN,2.498571,4.807218e+06,27644.723604,0.746705


In [329]:
df_gabung.describe(include="object")

,order_id,store_id,sku_id,customer_id,installer_id,installation_status,store_name,channel_type,city,store_tier,...,supplier_type,region,primary_family,secondary_family,quality_certification_status,partnership_start_date,return_id,return_reason,resolution_status,refund_method
count,321760,321760,321760,321760,204557,321760,321760,321760,280001,280001,...,309350,309350,309350,193674,309350,309350,24843,24843,24843,18919
unique,321760,40,1500,134274,1658,5,40,3,8,3,...,3,11,6,3,2,32,24843,5,3,3
top,DKR/230101/1013407,DKR-039,SKU-FURN-00646,CUSTB2Y2B20Z,TUKANG-JKT-990,Success,Dekoruma Mobile App,Physical Store,Alam Sutera,Standard,...,Pabrik,Klaten,Kursi,Lemari,Tersertifikasi,2021-08-03,RET-DKR/230114/8375,Failed Installation - Missing Parts,Selesai,Refund ke Rekening
freq,1,20986,266,29,166,175775,20986,280001,68955,178970,...,152612,61835,71783,85780,187713,25669,1,12637,18919,8863


In [330]:
df_gabung.isnull().sum()

order_id                             0
store_id                             0
sku_id                               0
customer_id                          0
order_date                           0
delivery_date                        0
installer_id                    117203
installation_date               117203
installation_status                  0
assembly_fee                         0
Install_time                    117203
delivery_time                        0
store_name                           0
channel_type                         0
city                             41759
store_tier                       41759
opening_date                         0
product_name                         0
category                             0
is_assembly_required                 0
supplier_id                      12410
unit_price                           0
product_family                       0
weight_category                      0
domicile_city                        0
registration_date        

In [331]:
(df_gabung.isnull().sum()/len(df_gabung)*100).round(2)

order_id                         0.00
store_id                         0.00
sku_id                           0.00
customer_id                      0.00
order_date                       0.00
delivery_date                    0.00
installer_id                    36.43
installation_date               36.43
installation_status              0.00
assembly_fee                     0.00
Install_time                    36.43
delivery_time                    0.00
store_name                       0.00
channel_type                     0.00
city                            12.98
store_tier                      12.98
opening_date                     0.00
product_name                     0.00
category                         0.00
is_assembly_required             0.00
supplier_id                      3.86
unit_price                       0.00
product_family                   0.00
weight_category                  0.00
domicile_city                    0.00
registration_date                0.00
registration

In [332]:
df_gabung = df_gabung.dropna(subset=['supplier_id'])

In [333]:
df_gabung['city'].value_counts()

city
Alam Sutera      66305
Kelapa Gading    57999
Bintaro          41559
Bekasi           33189
Kemang           32962
Surabaya         18634
Medan            11096
Bandung           7466
Name: count, dtype: int64

In [335]:
df_gabung['city'] = df_gabung['city'].fillna('Alam Sutera')

/var/folders/bb/_2z7z6bs4qbghjqy4r44sxkc0000gn/T/ipykernel_92981/3526205498.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gabung['city'] = df_gabung['city'].fillna('Alam Sutera')


In [334]:
df_gabung['store_tier'].value_counts()

store_tier
Standard    172135
Mini         64658
Flagship     32417
Name: count, dtype: int64

In [336]:
df_gabung['store_tier'] = df_gabung['store_tier'].fillna('Standard')

/var/folders/bb/_2z7z6bs4qbghjqy4r44sxkc0000gn/T/ipykernel_92981/1543353977.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gabung['store_tier'] = df_gabung['store_tier'].fillna('Standard')


In [337]:
df_gabung.to_csv('/Users/gustafdavid/Desktop/Purwadika/capston 2/gabungan.csv', index=False)